In [ ]:
# ===========================================
# Notebook Name:
# 03_build_deck_similarity
#
# Purpose:
# Calculate Pokemon-based similarity between
# unique functional deck compositions.
#
# Input:
# pokemon.gold.deck_pokemon_features
# pokemon.gold.deck_registry
#
# Output:
# pokemon.gold.deck_similarity
#
# Grain:
# One row per unique deck_hash pair
#
# Similarity:
# Weighted Jaccard using Pokemon quantities
#
# Business rules:
# - Staple Pokemon (played in a very high share
#   of all decks, e.g. generic search/support
#   Pokemon usable in almost any archetype) are
#   excluded before computing similarity. Without
#   this, decks cluster together mainly because
#   they share the same staples rather than the
#   archetype-defining Pokemon, since staples
#   dominate the intersection/union terms of the
#   Jaccard formula. The staple threshold is
#   computed dynamically from the same deck
#   population being compared (not hardcoded card
#   names), so it adapts automatically as the
#   metagame and card pool change.
# ===========================================

In [ ]:
from pyspark.sql import functions as F

DECK_POKEMON_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)

DECK_REGISTRY_TABLE = (
    "pokemon.gold.deck_registry"
)

DECK_SIMILARITY_TABLE = (
    "pokemon.gold.deck_similarity"
)

# A Pokemon played in more than this share of all
# decks is treated as a generic staple and excluded
# from the similarity calculation below (see the
# Business rules note in the header cell). 0.35 was
# chosen from the observed inclusion-rate distribution:
# as of 2026-07, there is a natural gap between the
# generic staples (Kecleon ex 61.5%, Meowth ex 44.5%,
# Grafaiai 44.1%, Poltchageist 39.0%) and the next most
# common, archetype-defining Pokemon (28.2% and below).
# Tune this if staples still dominate clusters (lower
# it) or if archetype-defining Pokemon are being
# excluded by mistake (raise it).
STAPLE_INCLUSION_RATE_THRESHOLD = 0.35

print("Input :", DECK_POKEMON_FEATURES_TABLE)
print("Input :", DECK_REGISTRY_TABLE)
print("Output:", DECK_SIMILARITY_TABLE)

In [0]:
features_df = spark.table(
    DECK_POKEMON_FEATURES_TABLE
)

registry_df = spark.table(
    DECK_REGISTRY_TABLE
)

print(
    "Feature rows:",
    features_df.count(),
)

print(
    "Unique decks:",
    features_df
    .select("deck_hash")
    .distinct()
    .count(),
)

In [ ]:
total_deck_count = (
    features_df
    .select("deck_hash")
    .distinct()
    .count()
)

pokemon_inclusion_df = (
    features_df
    .groupBy("card_name")
    .agg(
        F.countDistinct("deck_hash").alias(
            "decks_using_card"
        )
    )
    .withColumn(
        "inclusion_rate",
        F.col("decks_using_card")
        / F.lit(total_deck_count),
    )
)

staple_card_names_df = (
    pokemon_inclusion_df
    .filter(
        F.col("inclusion_rate")
        > STAPLE_INCLUSION_RATE_THRESHOLD
    )
    .select(
        "card_name",
        "decks_using_card",
        "inclusion_rate",
    )
)

staple_card_count = (
    staple_card_names_df.count()
)

print(
    "Deck population for inclusion rate:",
    total_deck_count,
)
print(
    "Staple Pokemon excluded from similarity "
    f"(inclusion_rate > "
    f"{STAPLE_INCLUSION_RATE_THRESHOLD:.0%}):",
    staple_card_count,
)

display(
    staple_card_names_df
    .orderBy(
        F.col("inclusion_rate").desc()
    )
)

In [ ]:
# similarity_features_df feeds only the pairwise
# similarity calculation below. features_df itself
# is left untouched so the debugging cells at the
# bottom of this notebook (and any other consumer of
# deck_pokemon_features) still see every Pokemon in
# the deck, including staples.
similarity_features_df = (
    features_df
    .join(
        staple_card_names_df.select("card_name"),
        on="card_name",
        how="left_anti",
    )
)

print(
    "Feature rows before staple exclusion:",
    features_df.count(),
)
print(
    "Feature rows after staple exclusion :",
    similarity_features_df.count(),
)

In [0]:
deck_list_df = (
    features_df
    .select("deck_hash")
    .distinct()
)

display(deck_list_df)

In [0]:
deck_pairs_df = (
    deck_list_df.alias("a")
    .crossJoin(
        deck_list_df.alias("b")
    )
    .filter(
        F.col("a.deck_hash")
        < F.col("b.deck_hash")
    )
    .select(
        F.col("a.deck_hash").alias(
            "deck_hash_a"
        ),
        F.col("b.deck_hash").alias(
            "deck_hash_b"
        ),
    )
)

display(deck_pairs_df)

In [0]:
deck_count = deck_list_df.count()
expected_pair_count = (
    deck_count * (deck_count - 1) // 2
)

actual_pair_count = (
    deck_pairs_df.count()
)

print("Deck count          :", deck_count)
print("Expected pair count :", expected_pair_count)
print("Actual pair count   :", actual_pair_count)

In [ ]:
features_a_df = (
    deck_pairs_df
    .join(
        similarity_features_df.select(
            F.col("deck_hash").alias(
                "feature_deck_hash"
            ),
            "card_name",
            "quantity",
        ),
        F.col("deck_hash_a")
        == F.col("feature_deck_hash"),
        "inner",
    )
    .select(
        "deck_hash_a",
        "deck_hash_b",
        "card_name",
        F.col("quantity")
        .cast("int")
        .alias("quantity_a"),
        F.lit(None)
        .cast("int")
        .alias("quantity_b"),
    )
)

display(features_a_df)

In [ ]:
features_b_df = (
    deck_pairs_df
    .join(
        similarity_features_df.select(
            F.col("deck_hash").alias(
                "feature_deck_hash"
            ),
            "card_name",
            "quantity",
        ),
        F.col("deck_hash_b")
        == F.col("feature_deck_hash"),
        "inner",
    )
    .select(
        "deck_hash_a",
        "deck_hash_b",
        "card_name",
        F.lit(None)
        .cast("int")
        .alias("quantity_a"),
        F.col("quantity")
        .cast("int")
        .alias("quantity_b"),
    )
)

display(features_b_df)

In [0]:
pair_card_quantities_df = (
    features_a_df
    .unionByName(features_b_df)
    .groupBy(
        "deck_hash_a",
        "deck_hash_b",
        "card_name",
    )
    .agg(
        F.max("quantity_a").alias(
            "quantity_a"
        ),
        F.max("quantity_b").alias(
            "quantity_b"
        ),
    )
    .fillna(
        0,
        subset=[
            "quantity_a",
            "quantity_b",
        ],
    )
    .withColumn(
        "intersection_quantity",
        F.least(
            F.col("quantity_a"),
            F.col("quantity_b"),
        ),
    )
    .withColumn(
        "union_quantity",
        F.greatest(
            F.col("quantity_a"),
            F.col("quantity_b"),
        ),
    )
    .withColumn(
        "is_shared_card",
        F.when(
            (F.col("quantity_a") > 0)
            & (F.col("quantity_b") > 0),
            1,
        ).otherwise(0),
    )
)

display(
    pair_card_quantities_df
    .orderBy(
        "deck_hash_a",
        "deck_hash_b",
        "card_name",
    )
)

In [0]:
deck_similarity_base_df = (
    pair_card_quantities_df
    .groupBy(
        "deck_hash_a",
        "deck_hash_b",
    )
    .agg(
        F.sum(
            "intersection_quantity"
        ).alias(
            "weighted_intersection"
        ),

        F.sum(
            "union_quantity"
        ).alias(
            "weighted_union"
        ),

        F.sum(
            "is_shared_card"
        ).alias(
            "shared_pokemon_names"
        ),

        F.count("*").alias(
            "union_pokemon_names"
        ),

        F.sum(
            F.abs(
                F.col("quantity_a")
                - F.col("quantity_b")
            )
        ).alias(
            "total_quantity_difference"
        ),
    )
    .withColumn(
        "weighted_jaccard_similarity",
        F.when(
            F.col("weighted_union") > 0,
            F.col("weighted_intersection")
            / F.col("weighted_union"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "presence_jaccard_similarity",
        F.when(
            F.col("union_pokemon_names") > 0,
            F.col("shared_pokemon_names")
            / F.col("union_pokemon_names"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "weighted_jaccard_pct",
        F.round(
            F.col(
                "weighted_jaccard_similarity"
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "presence_jaccard_pct",
        F.round(
            F.col(
                "presence_jaccard_similarity"
            ) * 100,
            2,
        ),
    )
)

In [0]:
registry_lookup_df = (
    registry_df
    .select(
        "deck_hash",
        "representative_deck_code",
        "best_rank",
        "average_rank",
        "result_appearance_count",
    )
    .dropDuplicates(["deck_hash"])
)

In [0]:
deck_similarity_df = (
    deck_similarity_base_df
    .join(
        registry_lookup_df.alias("a"),
        F.col("deck_hash_a")
        == F.col("a.deck_hash"),
        "left",
    )
    .join(
        registry_lookup_df.alias("b"),
        F.col("deck_hash_b")
        == F.col("b.deck_hash"),
        "left",
    )
    .select(
        "deck_hash_a",
        F.col(
            "a.representative_deck_code"
        ).alias("deck_code_a"),
        F.col(
            "a.best_rank"
        ).alias("best_rank_a"),
        F.col(
            "a.average_rank"
        ).alias("average_rank_a"),

        "deck_hash_b",
        F.col(
            "b.representative_deck_code"
        ).alias("deck_code_b"),
        F.col(
            "b.best_rank"
        ).alias("best_rank_b"),
        F.col(
            "b.average_rank"
        ).alias("average_rank_b"),

        "weighted_intersection",
        "weighted_union",
        "shared_pokemon_names",
        "union_pokemon_names",
        "total_quantity_difference",
        "weighted_jaccard_similarity",
        "weighted_jaccard_pct",
        "presence_jaccard_similarity",
        "presence_jaccard_pct",

        F.current_timestamp().alias(
            "updated_at"
        ),
    )
)

In [0]:
display(
    deck_similarity_df
    .orderBy(
        F.col(
            "weighted_jaccard_similarity"
        ).desc()
    )
)

In [0]:
similarity_pair_count = (
    deck_similarity_df.count()
)

if (
    similarity_pair_count
    != expected_pair_count
):
    raise ValueError(
        "Deck pair count mismatch. "
        f"expected={expected_pair_count}, "
        f"actual={similarity_pair_count}"
    )

print(
    "Validation passed: "
    "deck pair count is correct"
)

In [0]:
invalid_similarity_df = (
    deck_similarity_df
    .filter(
        (
            F.col(
                "weighted_jaccard_similarity"
            ) < 0
        )
        | (
            F.col(
                "weighted_jaccard_similarity"
            ) > 1
        )
        | (
            F.col(
                "presence_jaccard_similarity"
            ) < 0
        )
        | (
            F.col(
                "presence_jaccard_similarity"
            ) > 1
        )
    )
)

invalid_similarity_count = (
    invalid_similarity_df.count()
)

if invalid_similarity_count > 0:
    display(invalid_similarity_df)

    raise ValueError(
        f"{invalid_similarity_count} "
        "invalid similarities detected"
    )

print(
    "Validation passed: "
    "all similarities are between 0 and 1"
)

In [0]:
self_pair_count = (
    deck_similarity_df
    .filter(
        F.col("deck_hash_a")
        == F.col("deck_hash_b")
    )
    .count()
)

if self_pair_count > 0:
    raise ValueError(
        "Self-pairs were detected"
    )

print(
    "Validation passed: "
    "no self-pairs"
)

In [ ]:
# -------------------------------------------
# Rebuild deck_similarity atomically via CREATE
# OR REPLACE TABLE AS SELECT, instead of
# TRUNCATE + append. If the write fails partway,
# the old table contents are left intact rather
# than a table that has been emptied but not
# refilled.
# -------------------------------------------
deck_similarity_df.createOrReplaceTempView(
    "deck_similarity_staging"
)

spark.sql(f"""
CREATE OR REPLACE TABLE {DECK_SIMILARITY_TABLE}
COMMENT 'Pairwise Pokemon-based similarity between functional decks'
AS SELECT * FROM deck_similarity_staging
""")

for not_null_column in (
    "deck_hash_a",
    "deck_hash_b",
):
    spark.sql(
        f"ALTER TABLE {DECK_SIMILARITY_TABLE} "
        f"ALTER COLUMN {not_null_column} SET NOT NULL"
    )

print("Gold deck_similarity table rebuilt atomically.")

In [0]:
display(
    spark.table(
        DECK_SIMILARITY_TABLE
    )
    .select(
        "deck_code_a",
        "deck_code_b",
        "weighted_jaccard_pct",
        "presence_jaccard_pct",
        "shared_pokemon_names",
        "union_pokemon_names",
        "total_quantity_difference",
    )
    .orderBy(
        F.col(
            "weighted_jaccard_pct"
        ).desc()
    )
    .limit(20)
)

In [0]:
TARGET_DECK_CODE = "MXyypR-czbVrd-pMEMRR"

similar_to_target_df = (
    spark.table(
        DECK_SIMILARITY_TABLE
    )
    .filter(
        (F.col("deck_code_a")
         == TARGET_DECK_CODE)
        | (F.col("deck_code_b")
           == TARGET_DECK_CODE)
    )
    .withColumn(
        "similar_deck_code",
        F.when(
            F.col("deck_code_a")
            == TARGET_DECK_CODE,
            F.col("deck_code_b"),
        ).otherwise(
            F.col("deck_code_a")
        ),
    )
    .withColumn(
        "similar_deck_hash",
        F.when(
            F.col("deck_code_a")
            == TARGET_DECK_CODE,
            F.col("deck_hash_b"),
        ).otherwise(
            F.col("deck_hash_a")
        ),
    )
    .select(
        F.lit(
            TARGET_DECK_CODE
        ).alias("target_deck_code"),
        "similar_deck_code",
        "similar_deck_hash",
        "weighted_jaccard_pct",
        "presence_jaccard_pct",
        "shared_pokemon_names",
        "union_pokemon_names",
        "total_quantity_difference",
    )
    .orderBy(
        F.col(
            "weighted_jaccard_pct"
        ).desc()
    )
)

display(similar_to_target_df)

In [0]:
COMPARISON_DECK_CODE = (
    similar_to_target_df
    .select("similar_deck_code")
    .first()["similar_deck_code"]
)

print(
    "Target     :",
    TARGET_DECK_CODE,
)

print(
    "Comparison :",
    COMPARISON_DECK_CODE,
)

In [0]:
target_features_df = (
    features_df
    .filter(
        F.col(
            "representative_deck_code"
        ) == TARGET_DECK_CODE
    )
    .select(
        "card_name",
        F.col("quantity").alias(
            "target_quantity"
        ),
    )
)

comparison_features_df = (
    features_df
    .filter(
        F.col(
            "representative_deck_code"
        ) == COMPARISON_DECK_CODE
    )
    .select(
        "card_name",
        F.col("quantity").alias(
            "comparison_quantity"
        ),
    )
)

deck_difference_df = (
    target_features_df
    .join(
        comparison_features_df,
        on="card_name",
        how="full",
    )
    .fillna(
        0,
        subset=[
            "target_quantity",
            "comparison_quantity",
        ],
    )
    .withColumn(
        "quantity_difference",
        F.col("target_quantity")
        - F.col("comparison_quantity"),
    )
    .orderBy(
        F.abs(
            F.col("quantity_difference")
        ).desc(),
        "card_name",
    )
)

display(deck_difference_df)